# Statystyka dla Data Analyst/Engineer — Moduł 9: Testy chi-kwadrat

Testy t z Modułu 8 porównują ŚREDNIE zmiennych ciągłych. Ale co, gdy interesują nas
zmienne **kategoryczne** — miasto, kategoria produktu, grupa kampanii, status zamówienia?
Rodzina **testów chi-kwadrat** (χ²) odpowiada na dwa pokrewne pytania: czy dwie
kategoryczne cechy są ze sobą powiązane (**test niezależności**), i czy obserwowany
rozkład kategorii pasuje do oczekiwanego (**test zgodności**).

## Spis treści
1. [Tabele kontyngencji (crosstab)](#sec1)
2. [Test niezależności chi-kwadrat](#sec2)
3. [Założenia testu i mała liczebność komórek](#sec3)
4. [Test zgodności chi-kwadrat](#sec4)
5. [Wielkość efektu: V Craméra](#sec5)
6. [Podsumowanie i ćwiczenia](#sec6)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Tabele kontyngencji (crosstab)

**Tabela kontyngencji** (*contingency table*, krzyżowa) zestawia liczności dla każdej
kombinacji dwóch zmiennych kategorycznych. To punkt wyjścia dla każdego testu
chi-kwadrat niezależności. Zbudujmy tabelę: grupa kampanii (A/B) x konwersja (tak/nie) —
dokładnie te dane, które w Module 3 i 7 analizowaliśmy inaczej.

In [ ]:
tabela = pd.crosstab(klienci["grupa_kampanii"], klienci["konwersja"])
tabela


`pd.crosstab` z argumentem `normalize="index"` pokaże od razu proporcje w wierszach —
wygodne do szybkiego porównania wizualnego przed formalnym testem:

In [ ]:
pd.crosstab(klienci["grupa_kampanii"], klienci["konwersja"], normalize="index")


<a id="sec2"></a>
## 2. Test niezależności chi-kwadrat

**Test niezależności chi-kwadrat** sprawdza, czy dwie zmienne kategoryczne są od siebie
niezależne (Moduł 3!).

- H0: grupa kampanii i konwersja są **niezależne** (brak związku).
- H1: grupa kampanii i konwersja są **zależne** (istnieje związek).

Statystyka testowa porównuje zaobserwowane liczności z licznościami, jakich
oczekiwalibyśmy, GDYBY zmienne faktycznie były niezależne (czyli gdyby proporcje
konwersji były identyczne w obu grupach).

In [ ]:
from scipy import stats

chi2, wartosc_p, df, oczekiwane = stats.chi2_contingency(tabela)

print(f"Statystyka chi2: {chi2:.3f}")
print(f"Stopnie swobody: {df}")
print(f"Wartosc p: {wartosc_p:.4f}")
print("Oczekiwane liczebnosci (gdyby H0 byla prawdziwa):")
print(oczekiwane)


In [ ]:
alpha = 0.05
if wartosc_p <= alpha:
    print("Odrzucamy H0 -- grupa kampanii i konwersja NIE sa niezalezne")
else:
    print("Nie odrzucamy H0 -- brak dowodow na zaleznosc")


> 📊 **Chi-kwadrat to formalizacja tego, co widzieliśmy nieformalnie**
>
> To dokładnie ten sam efekt A/B, który w Module 3 sprawdzaliśmy prawdopodobieństwem warunkowym, w Module 6 porównywaliśmy przedziałami ufności proporcji, a w Module 7 testowaliśmy testem z dla jednej proporcji. Test chi-kwadrat niezależności to naturalne narzędzie, gdy masz DWIE grupy do porównania na zmiennej kategorycznej — i uogólnia się bez trudu na WIĘCEJ niż dwie grupy naraz (np. konwersja wg miasta, nie tylko wg dwóch grup A/B).

<a id="sec3"></a>
## 3. Założenia testu i mała liczebność komórek

Test chi-kwadrat opiera się na przybliżeniu i wymaga, żeby **oczekiwana liczebność**
(wartości z `oczekiwane` powyżej) w każdej komórce tabeli wynosiła **co najmniej 5**.
Przy mniejszych oczekiwanych licznościach przybliżenie chi-kwadrat staje się niewiarygodne.

In [ ]:
minimalna_oczekiwana = oczekiwane.min()
print(f"Najmniejsza oczekiwana liczebnosc w tabeli: {minimalna_oczekiwana:.2f}")
print("Zalozenie spelnione" if minimalna_oczekiwana >= 5 else "UWAGA: zalozenie NIE jest spelnione")


> ⚠️ **Gdy liczebności są za małe: dokładny test Fishera**
>
> Dla małych tabel (zwłaszcza `2x2`) z niską oczekiwaną licznością lepszym wyborem jest **dokładny test Fishera** (`stats.fisher_exact`) — liczy dokładne prawdopodobieństwo zamiast polegać na przybliżeniu chi-kwadrat. Jest wolniejszy obliczeniowo dla dużych tabel, ale dokładny nawet przy bardzo małych licznościach.

In [ ]:
# Dokladny test Fishera dziala tylko na tabelach 2x2
odds_ratio, p_fisher = stats.fisher_exact(tabela)
print(f"Iloraz szans: {odds_ratio:.3f}, wartosc p (Fisher): {p_fisher:.4f}")
print(f"(dla porownania, wartosc p chi-kwadrat: {wartosc_p:.4f})")


<a id="sec4"></a>
## 4. Test zgodności chi-kwadrat

**Test zgodności** (*goodness-of-fit*) sprawdza, czy obserwowany rozkład JEDNEJ zmiennej
kategorycznej pasuje do zakładanego, oczekiwanego rozkładu. Przypomnij sobie z kodu
generującego dane: kolumna `miasto` została wylosowana z zadanymi proporcjami
(Warszawa 30%, Kraków 25%, Gdańsk/Wrocław/Poznań po 15%). Sprawdźmy, czy obserwowana
próbka faktycznie pasuje do tych założonych proporcji.

- H0: rozkład miast w próbce jest zgodny z zakładanymi proporcjami.
- H1: rozkład miast różni się od zakładanych proporcji.

In [ ]:
zakladane_proporcje = {"Warszawa": 0.30, "Krakow": 0.25, "Gdansk": 0.15, "Wroclaw": 0.15, "Poznan": 0.15}

obserwowane = klienci["miasto"].value_counts().reindex(zakladane_proporcje.keys())
n = len(klienci)
oczekiwane_gof = pd.Series(zakladane_proporcje) * n

print("Obserwowane:")
print(obserwowane)
print("\nOczekiwane pod H0:")
print(oczekiwane_gof)

chi2_gof, wartosc_p_gof = stats.chisquare(f_obs=obserwowane, f_exp=oczekiwane_gof)
print(f"\nchi2 = {chi2_gof:.3f}, wartosc p = {wartosc_p_gof:.4f}")


Skoro dane faktycznie wygenerowaliśmy z tych właśnie proporcji, spodziewaj się dużej
wartości p (brak podstaw do odrzucenia H0) — obserwowane różnice to tylko losowy szum
próbki 400 klientów wokół prawdziwych proporcji.

<a id="sec5"></a>
## 5. Wielkość efektu: V Craméra

Podobnie jak d Cohena dla testów t (Moduł 8), **V Craméra** mierzy SIŁĘ związku między
dwiema zmiennymi kategorycznymi, niezależnie od wielkości próby (a więc niezależnie od
tego, czy test wyszedł "istotny"):

```
V = sqrt(chi2 / (n * (min(k, r) - 1)))
```

gdzie `k` i `r` to liczba kolumn i wierszy tabeli. `V` mieści się w przedziale `[0, 1]` —
`0` oznacza brak związku, `1` oznacza związek doskonały. Orientacyjne progi (podobnie
umowne jak przy d Cohena): `0.1` mały, `0.3` średni, `0.5`+ duży efekt (dla tabeli 2x2).

In [ ]:
def v_cramera(tabela_kontyngencji):
    chi2_v, _, _, _ = stats.chi2_contingency(tabela_kontyngencji)
    n_v = tabela_kontyngencji.sum().sum()
    min_wymiar = min(tabela_kontyngencji.shape) - 1
    return (chi2_v / (n_v * min_wymiar)) ** 0.5


v = v_cramera(tabela)
print(f"V Cramera (grupa kampanii x konwersja): {v:.3f}")


<a id="sec6"></a>
## 6. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- tabele kontyngencji (`pd.crosstab`) jako punkt wyjścia do analizy zmiennych
  kategorycznych,
- test niezależności chi-kwadrat (`stats.chi2_contingency`),
- założenie o minimalnej oczekiwanej liczebności komórek i alternatywę — dokładny test
  Fishera,
- test zgodności chi-kwadrat (`stats.chisquare`) dla jednej zmiennej,
- V Craméra jako miarę siły związku, niezależną od wielkości próby.

> 📝 **Ćwiczenie 1: Region a status zamówienia**
>
> Zbuduj tabelę kontyngencji `region` x `konwersja` dla wszystkich 4 regionów (nie tylko A/B kampanii) i wykonaj test niezależności chi-kwadrat. Czy region wydaje się być powiązany z konwersją?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
tabela_regionow = pd.crosstab(klienci["region"], klienci["konwersja"])
chi2_r, p_r, df_r, oczek_r = stats.chi2_contingency(tabela_regionow)
print(tabela_regionow)
print(f"chi2={chi2_r:.3f}, df={df_r}, p={p_r:.4f}")
```

Konwersja w naszych danych zalezy TYLKO od grupy kampanii, nie od regionu -- spodziewaj sie wartosci p powyzej 0.05 (czyli braku istotnosci), choc moze wyjsc dosc blisko progu -- to normalne, przypadkowy szum probki czasem zblizy sie do progu istotnosci nawet bez prawdziwego efektu.
</details>

> 📝 **Ćwiczenie 2: Sprawdzenie liczebności oczekiwanych**
>
> Dla tabeli z ćwiczenia 1, sprawdź minimalną oczekiwaną liczebność komórek. Czy założenie `>= 5` jest spełnione dla tej tabeli?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
print(oczek_r.min())
print("Zalozenie spelnione" if oczek_r.min() >= 5 else "Zalozenie NIE spelnione")
```
</details>

> 📝 **Ćwiczenie 3: Zgodność rozkładu grupy kampanii**
>
> Kod generujący dane losuje `grupa_kampanii` z równymi proporcjami 50/50. Wykonaj test zgodności chi-kwadrat sprawdzający, czy obserwowany podział na grupy A/B w naszej próbce jest zgodny z proporcjami 50/50.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
obserwowane_grup = klienci["grupa_kampanii"].value_counts().reindex(["A", "B"])
oczekiwane_grup = pd.Series([0.5, 0.5], index=["A", "B"]) * len(klienci)
chi2_g, p_g = stats.chisquare(f_obs=obserwowane_grup, f_exp=oczekiwane_grup)
print(f"chi2={chi2_g:.3f}, p={p_g:.4f}")
```
</details>

> 📝 **Ćwiczenie 4: V Craméra dla regionów**
>
> Policz V Craméra dla tabeli `region` x `konwersja` z ćwiczenia 1, korzystając z funkcji `v_cramera` z sekcji 5. Porównaj z V Craméra dla grupy kampanii x konwersja z sekcji 5 -- które powiązanie jest silniejsze?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
v_regionow = v_cramera(tabela_regionow)
print(f"V Cramera (region x konwersja): {v_regionow:.3f}")
print(f"V Cramera (grupa kampanii x konwersja): {v:.3f}")
```

Zaskakujaco, oba V moga wyjsc bardzo zblizone, mimo ze test dla regionu (cwiczenie 1) NIE byl istotny statystycznie, a dla grupy kampanii (sekcja 2) byl. To wazna lekcja: V Cramera zalezy tylko od chi2/n/wymiarow tabeli, a nie od liczby stopni swobody wprost tak jak wartosc p -- przy wiekszej tabeli (4 regiony) sam szum probki latwiej wygeneruje umiarkowane V, mimo braku prawdziwego zwiazku. Efekt i istotnosc trzeba zawsze czytac razem, nie osobno.
</details>

> 🔥 **Wyzwanie: symulacyjna weryfikacja testu niezależności**
>
> Zweryfikuj test chi-kwadrat niezależności z sekcji 2 metodą permutacyjną: 10 000 razy losowo potasuj kolumnę `konwersja` (niezależnie od `grupa_kampanii`), za każdym razem licząc statystykę chi-kwadrat dla przetasowanej tabeli (`stats.chi2_contingency(pd.crosstab(klienci['grupa_kampanii'], przetasowana_konwersja))[0]`). Policz odsetek permutacji, w których statystyka permutowana >= statystyka oryginalna -- to Twoja permutacyjna wartość p. Porównaj z wynikiem z sekcji 2.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import numpy as np

rng = np.random.default_rng(4)
liczba_permutacji = 2000
liczba_skrajnych = 0

for _ in range(liczba_permutacji):
    przetasowana = rng.permutation(klienci["konwersja"].values)
    tabela_sym = pd.crosstab(klienci["grupa_kampanii"], przetasowana)
    chi2_sym, _, _, _ = stats.chi2_contingency(tabela_sym)
    if chi2_sym >= chi2:
        liczba_skrajnych += 1

p_permutacyjne = liczba_skrajnych / liczba_permutacji
print(f"Wartosc p (permutacyjna): {p_permutacyjne:.4f}")
print(f"Wartosc p (chi2_contingency, sekcja 2): {wartosc_p:.4f}")
```

Permutacja niszczy CELOWO wszelki zwiazek miedzy kolumnami (bo tasujemy jedna z nich niezaleznie od drugiej), wiec symuluje dokladnie to, co zaklada H0 -- podobienstwo do wyniku formalnego testu to dodatkowe potwierdzenie jego poprawnosci.
</details>

## Co dalej?

Testy t (Moduł 8) porównują dwie średnie, a testy chi-kwadrat (ten moduł) — kategorie. Co
gdy chcemy porównać średnie w **więcej niż dwóch** grupach naraz (np. satysfakcję we
wszystkich czterech regionach)? W **Module 10** poznamy **ANOVA** — analizę wariancji,
zaprojektowaną dokładnie do tego zadania, i formalnie sprawdzimy tę samą różnicę
regionów, którą zobaczyliśmy wizualnie w Module 2.